# Cell Neighborhood Tokenization

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 23.04.2024
- Date of Last Modification: 28.06.2024

## 1. Setup

### 1.1 Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import pickle
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scgpt as scg
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
import torch
from datasets import load_from_disk
from pyensembl import EnsemblRelease
from scipy.stats import spearmanr
from scgpt.model import TransformerModel
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from skmisc.loess import loess

from nichejepa import tokenizers
from nichejepa import utils

/home/aih/sebastian.birk/miniconda3/envs/nichejepa/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/home/aih/sebastian.birk/miniconda3/envs/nichejepa/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


### 1.2 Run Notebook Setup

In [3]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### 1.3 Configure Paths and Create Directories

In [4]:
# Define paths
st_data_folder_path = "../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
tokenizer = "cell_neighborhood_tokenizer"
tokenizer_folder_path = f"../datasets/tokenizer/{tokenizer}"
results_folder_path = "../datasets/results"
geneformer_token_dictionary_file = f"../datasets/tokenizer/geneformer_token_dictionary.pkl"
scgpt_model_dir = Path("../artifacts/scgpt/pretrained_model")
scgpt_model_config_file = scgpt_model_dir / "args.json"
scgpt_model_file = scgpt_model_dir / "best_model.pt"
scgpt_vocab_file = scgpt_model_dir / "vocab.json"
scgpt_metadata_file = Path("../artifacts/scgpt/metadata/gene_info.csv")

## 2. Tokenization

In [5]:
# Define dataset
dataset_name = "nanostring_cosmx_human_brain"
#dataset_name = "sim_ref_1354fts_10000locs_8r_8c_20p_3fc"
#dataset_name = "sim_free_1354fts_10000locs_8r_8c_20p_3fc"
#dataset_name = "merfish_mouse_hypothalamus"
#dataset_name = "xenium_human_brain_cancer"

In [6]:
add_cls_tokens = False
separate_cell_and_neighborhood_tokens = False

if dataset_name == "nanostring_cosmx_human_brain":
    n_batches = 1
    species = "homo_sapiens"
    norm_factors = ["read_depth", "cell_area", "unnormalized"]
    custom_attr_name_dict = {"cell_type": "cell_types",
                             "niche_type": "niche_types"}
    label_cats = ["cell_types", "niche_types"]
elif dataset_name == "merfish_mouse_hypothalamus":
    n_batches = 5
    species = "mus_musculus"
    norm_factors = ["prenormalized"] # already normalized by cell volume
    custom_attr_name_dict = {"cell_type": "cell_types",
                             "niche_type": "niche_types"}
    label_cats = ["cell_types", "niche_types"]
elif dataset_name == "xenium_human_brain_cancer":
    n_batches = 1
    species = "homo_sapiens"
    norm_factors = ["read_depth", "cell_area"]
    custom_attr_name_dict = {"cluster": "cluster"}
    label_cats = ["cluster"]
elif "sim" in dataset_name:
    n_batches = 1
    species = "homo_sapiens"
    norm_factors = ["read_depth", "unnormalized"]
    custom_attr_name_dict = {"cell_types": "cell_types",
                             "niche_types": "niche_types"}
    label_cats = ["cell_types", "niche_types"]

os.makedirs(f"{tokenizer_folder_path}/{dataset_name}", exist_ok=True)

# Set dataset-specific paths and parameters
cell_gene_means_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_means_dictionary.pkl"
cell_gene_nzmedians_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_nzmedians_dictionary.pkl"
cell_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_logmeans_dictionary.pkl"
neighborhood_gene_means_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_means_dictionary.pkl"
neighborhood_gene_nzmedians_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_nzmedians_dictionary.pkl"
neighborhood_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_logmeans_dictionary.pkl"
nichejepa_token_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/token_dictionary" \
                                  f"{'_cls' if add_cls_tokens else '_no_cls'}" \
                                  f"{'_sep' if separate_cell_and_neighborhood_tokens else '_no_sep'}.pkl"
token_dictionary_file = f"../datasets/tokenizer/token_dictionary_{species}.pkl"
input_directory = f"{st_data_silver_folder_path}/{dataset_name}"
output_directory = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}"

### 2.1 Create Token Dictionary

In [7]:
if species == "mus_musculus":
    # Extract Ensembl IDs of protein coding and miRNA mouse genes
    ensembl = EnsemblRelease(111, species=species)
    ensembl.download()
    ensembl.index()
    all_genes = ensembl.genes()
    protein_coding_genes = [gene for gene in all_genes if gene.biotype == "protein_coding"]
    mirna_genes = [gene for gene in all_genes if gene.biotype == "miRNA"]
    all_relevant_genes = protein_coding_genes + mirna_genes
    gene_ids = [gene.gene_id for gene in all_relevant_genes]
    
    # Create token dictionary for mouse
    token_dict = {'<pad>': 0, '<mask>': 1}
    token_dict.update({gene_id: idx + 2 for idx, gene_id in enumerate(gene_ids)})
    with open(token_dictionary_file, "wb") as f:
        pickle.dump(token_dict, f)
elif species == "homo_sapiens":
    # Load geneformer token dict to get protein-coding and miRNA genes for human
    with open(geneformer_token_dictionary_file, "rb") as f:
        token_dict = pickle.load(f)
    with open(token_dictionary_file, "wb") as f:
            pickle.dump(token_dict, f)

In [8]:
# Load data
if n_batches == 1:
    adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")
else:
    adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch1.h5ad")

In [9]:
### Create token dict for nichejepa ###
with open(token_dictionary_file, "rb") as f:
    token_dict = pickle.load(f)

# Add cls token
if add_cls_tokens:
    def insert_at_index(d, index, key, value):
        items = list(d.items())
        items.insert(index, (key, value))
        new_d = dict(items)
        return new_d
    token_dict = insert_at_index(token_dict, 2, "<cls_cell>", 2)
    if separate_cell_and_neighborhood_tokens:
        token_dict = insert_at_index(token_dict, 3, "<cls_neighborhood>", 3)

# Remove genes not in dataset
gene_list = adata.var["ensembl_id"].tolist()
token_dict_copy = token_dict.copy()
for token in token_dict_copy.keys():
    if "ENS" in token and token not in gene_list:
        del(token_dict[token])
del(token_dict_copy)

# Separate gene tokens into cell and neighborhood tokens
if separate_cell_and_neighborhood_tokens:
    new_token_dict = {
        (key + "_cell" if "ENS" in key else key): value
        for value, key in enumerate(token_dict.keys())}
    new_token_dict_copy = new_token_dict.copy()
    i = 0
    for key in new_token_dict_copy.keys():
        if "ENS" in key:
            new_token_dict[key.replace("cell", "neighborhood")] = len(new_token_dict_copy) + i
            i += 1
    del(new_token_dict_copy)
else:
    new_token_dict = {
        key: value
        for value, key in enumerate(token_dict.keys())}

# Write to disk
with open(nichejepa_token_dictionary_file, "wb") as file:
    pickle.dump(new_token_dict, file)

In [10]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

In [11]:
# NicheJEPA token dict
len(token_dict)

6033

In [12]:
n_gene_tokens = len([token for token in token_dict.keys() if "ENS" in token])

In [13]:
# First n tokens
n = 5
for key, value in list(token_dict.items())[:n]:
    print(f"{key}: {value}")

<pad>: 0
<mask>: 1
ENSG00000000938: 2
ENSG00000000971: 3
ENSG00000001036: 4


In [14]:
if separate_cell_and_neighborhood_tokens and species == "homo_sapiens":
    # First cell gene token
    print(token_dict["ENSG00000000938_cell"])
    # First neighborhood gene token
    print(token_dict["ENSG00000000938_neighborhood"])

### 2.2 Compute Normalization Factors for Gene Counts

Corpus means, non-zero medians, or log means are used depending on the 'norm_method' specified in the CellNeighborhoodRankTokenizer (check source code).

In [33]:
# Compute Corpus Means and Non-zero Medians for Cell and Neighborhood Genes
for norm_factor in norm_factors:
    # Load data
    if n_batches == 1:
        adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")
        if type(adata.X) == sp._csr.csr_matrix:
            adata.X = adata.X.toarray()
    
        # Filter cells that did not pass QC
        if "filter_pass" in adata.obs.columns:
            filter_pass_idx = np.where([filter_pass == 1 for filter_pass in adata.obs["filter_pass"]])[0]
        else:
            print(f"adata has no column 'filter_pass'; utilizing all cells.")
            filter_pass_idx = np.array([i for i in range(adata.shape[0])])
        adata = adata[filter_pass_idx].copy()
        
        # Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
        sq.gr.spatial_neighbors(adata,
                                coord_type="generic",
                                spatial_key="spatial",
                                radius=27.5,
                                )
        
        # Add self loops to spatial neighborhood graph
        adata.obsp["spatial_connectivities"].setdiag(1)
            
        # Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
        adata.layers["X_neighborhood"] = adata.obsp["spatial_connectivities"].T @ adata.X

        if norm_factor == "read_depth":
            # Normalize cell gene counts by read depth
            adata.X = adata.X / adata.X.sum(1).reshape(-1, 1) * 10000
        
            # Normalize neighborhood gene counts by read depth
            adata.layers["X_neighborhood"] = (
                adata.layers["X_neighborhood"] /
                adata.layers["X_neighborhood"].sum(1).reshape(-1, 1) * 10000)
        elif norm_factor == "cell_area":
            # Normalize cell gene counts by cell area
            adata.X = adata.X / adata.obs["cell_area"].values.reshape(-1, 1)
        
            # Compute neighborhood area
            adata.obs["neighborhood_cell_area"] = np.array(adata.obsp["spatial_connectivities"].T @ adata.obs["cell_area"].values.reshape(-1, 1))  
        
            # Normalize neighborhood counts
            adata.layers["X_neighborhood"] = (
                adata.layers["X_neighborhood"] /
                adata.obs["neighborhood_cell_area"].values.reshape(-1, 1))
        elif norm_factor == "prenormalized":
            pass
        elif norm_factor == "unnormalized":
            pass
    else:
        adata_batch_list = []
        for batch_idx in range(1, n_batches+1):
            adata_batch = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch{batch_idx}.h5ad")

            # Filter cells that did not pass QC
            if "filter_pass" in adata_batch.obs.columns:
                filter_pass_idx = np.where([filter_pass == 1 for filter_pass in adata_batch.obs["filter_pass"]])[0]
            else:
                print(f"adata has no column 'filter_pass'; utilizing all cells.")
                filter_pass_idx = np.array([i for i in range(adata_batch.shape[0])])
            adata_batch = adata_batch[filter_pass_idx].copy()

            # Compute spatial neighborhood graph based on Visium spot diameter of 55 microns
            sq.gr.spatial_neighbors(adata_batch,
                                    coord_type="generic",
                                    spatial_key="spatial",
                                    radius=27.5,
                                    )
    
            # Add self loops to spatial neighborhood graph
            adata_batch.obsp["spatial_connectivities"].setdiag(1)
                
            # Compute sum of counts across each cell's neighborhood to get neighborhood counts per cell
            adata_batch.layers["X_neighborhood"] = adata_batch.obsp["spatial_connectivities"].T @ adata_batch.X

            if norm_factor == "read_depth":
                # Normalize cell gene counts by read depth
                adata_batch.X = adata_batch.X / adata_batch.X.sum(1).reshape(-1, 1) * 10000
            
                # Normalize neighborhood gene counts by read depth
                adata_batch.layers["X_neighborhood"] = (
                    adata_batch.layers["X_neighborhood"] /
                    adata_batch.layers["X_neighborhood"].sum(1).reshape(-1, 1) * 10000)
            elif norm_factor == "cell_area":
                # Normalize cell gene counts by cell area
                adata_batch.X = adata_batch.X / adata_batch.obs["cell_area"].values.reshape(-1, 1)
            
                # Compute neighborhood area
                adata_batch.obs["neighborhood_cell_area"] = np.array(adata_batch.obsp["spatial_connectivities"].T @ adata_batch.obs["cell_area"].values.reshape(-1, 1))  
            
                # Normalize neighborhood counts
                adata_batch.layers["X_neighborhood"] = (
                    adata_batch.layers["X_neighborhood"] /
                    adata_batch.obs["neighborhood_cell_area"].values.reshape(-1, 1))
            elif norm_factor == "prenormalized":
                pass
            elif norm_factor == "unnormalized":
                pass
                    
            adata_batch_list.append(adata_batch)
        adata = ad.concat(adata_batch_list, join="inner", axis=0)

        # Combine spatial neighborhood graphs as disconnected components
        batch_connectivities = []
        len_before_batch = 0
        for i in range(len(adata_batch_list)):
            if i == 0: # first batch
                after_batch_connectivities_extension = sp.csr_matrix(
                    (adata_batch_list[0].shape[0],
                    (adata.shape[0] -
                    adata_batch_list[0].shape[0])))
                batch_connectivities.append(sp.hstack(
                    (adata_batch_list[0].obsp["spatial_connectivities"],
                    after_batch_connectivities_extension)))
            elif i == (len(adata_batch_list) - 1): # last batch
                before_batch_connectivities_extension = sp.csr_matrix(
                    (adata_batch_list[i].shape[0],
                    (adata.shape[0] -
                    adata_batch_list[i].shape[0])))
                batch_connectivities.append(sp.hstack(
                    (before_batch_connectivities_extension,
                    adata_batch_list[i].obsp["spatial_connectivities"])))
            else: # middle batches
                before_batch_connectivities_extension = sp.csr_matrix(
                    (adata_batch_list[i].shape[0], len_before_batch))
                after_batch_connectivities_extension = sp.csr_matrix(
                    (adata_batch_list[i].shape[0],
                    (adata.shape[0] -
                    adata_batch_list[i].shape[0] -
                    len_before_batch)))
                batch_connectivities.append(sp.hstack(
                    (before_batch_connectivities_extension,
                    adata_batch_list[i].obsp["spatial_connectivities"],
                    after_batch_connectivities_extension)))
            len_before_batch += adata_batch_list[i].shape[0]
        adata.obsp["spatial_connectivities"] = sp.vstack(batch_connectivities)

        # Copy ensembl IDs
        adata.var = adata_batch_list[0].var
        
    # Compute means and non-zero medians
    cell_gene_means = np.mean(adata.X, axis=0)
    masked_data = np.ma.masked_equal(adata.X, 0)
    cell_gene_nzmedians = np.ma.mean(masked_data, axis=0)
    cell_gene_nzmedians = cell_gene_nzmedians.filled(0)
    neighborhood_gene_means = np.mean(adata.layers["X_neighborhood"], axis=0)
    masked_data_neighborhood = np.ma.masked_equal(adata.layers["X_neighborhood"], 0)
    neighborhood_gene_nzmedians = np.ma.mean(masked_data_neighborhood, axis=0)
    neighborhood_gene_nzmedians = neighborhood_gene_nzmedians.filled(0)

    sc.pp.log1p(adata)
    sc.pp.log1p(adata, layer="X_neighborhood")
    cell_gene_logmeans = np.mean(adata.X, axis=0)
    neighborhood_gene_logmeans = np.mean(adata.layers["X_neighborhood"], axis=0)

    cell_gene_logmeans = np.array(cell_gene_logmeans).reshape(-1)
    cell_gene_nzmedians = np.array(cell_gene_nzmedians).reshape(-1)
    cell_gene_means = np.array(cell_gene_means).reshape(-1)
    neighborhood_gene_means = np.array(neighborhood_gene_means).reshape(-1)
    neighborhood_gene_nzmedians = np.array(neighborhood_gene_nzmedians).reshape(-1)
    neighborhood_gene_logmeans = np.array(neighborhood_gene_logmeans).reshape(-1)
    
    # Create norm factor dictionaries for tokenizer and store on disk
    cell_gene_means_dict = {}
    cell_gene_nzmedians_dict = {}
    cell_gene_logmeans_dict = {}
    neighborhood_gene_means_dict = {}
    neighborhood_gene_nzmedians_dict = {}
    neighborhood_gene_logmeans_dict = {}
    
    for (gene,
         cell_gene_mean,
         cell_gene_nzmedian,
         cell_gene_logmean,
         neighborhood_gene_mean,
         neighborhood_gene_nzmedian,
         neighborhood_gene_logmean) in zip(
            adata.var["ensembl_id"].values,
            cell_gene_means,
            cell_gene_nzmedians,
            cell_gene_logmeans,
            neighborhood_gene_means,
            neighborhood_gene_nzmedians,
            neighborhood_gene_logmeans):
        cell_gene_means_dict[gene] = cell_gene_mean
        cell_gene_nzmedians_dict[gene] = cell_gene_nzmedian
        cell_gene_logmeans_dict[gene] = cell_gene_logmean
        neighborhood_gene_means_dict[gene] = neighborhood_gene_mean
        neighborhood_gene_nzmedians_dict[gene] = neighborhood_gene_nzmedian
        neighborhood_gene_logmeans_dict[gene] = neighborhood_gene_logmean
    
    with open(cell_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_means_dict, file)
    with open(cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_nzmedians_dict, file)
    with open(cell_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(cell_gene_logmeans_dict, file)
    with open(neighborhood_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_means_dict, file)
    with open(neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_nzmedians_dict, file)
    with open(neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
        pickle.dump(neighborhood_gene_logmeans_dict, file)

adata has no column 'filter_pass'; utilizing all cells.
adata has no column 'filter_pass'; utilizing all cells.


In [34]:
values = []
for key, value in cell_gene_means_dict.items():
    values.append(value)

np.mean(values)

1.0180580502215657

In [35]:
values = []
for key, value in neighborhood_gene_means_dict.items():
    values.append(value)

np.mean(values)

6.242840694239291

### 2.3 Run Tokenizer

In [ ]:
# Tokenize data with different normalization methods
for model_input_size in [n_gene_tokens * 2, # all genes
                         #512,
                        ]:
    for norm_method in ["analytic_pearson_residuals",
                        "mean",
                        "nzmedian",
                        "seurat_v3",
                        "shifted_logmean",
                        "shifted_log",
    ]:
        if norm_method == "analytic_pearson_residuals":
            output_file_prefix = f"{dataset_name}_{norm_method}_{model_input_size}"
    
            tk = tokenizers.CellNeighborhoodRankTokenizer(
                    model_input_size=model_input_size,
                    norm_method=norm_method,
                    separate_cell_and_neighborhood_tokens=separate_cell_and_neighborhood_tokens,
                    cell_gene_means_file=cell_gene_means_dictionary_file.replace("gene", f"gene_read_depth"),
                    cell_gene_nzmedians_file=cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_read_depth"),
                    cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_nzmedians_file=neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_read_depth"),
                    neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_read_depth"),
                    token_dictionary_file=nichejepa_token_dictionary_file,
                    custom_attr_name_dict=custom_attr_name_dict)
    
            tk.tokenize_data(input_directory=input_directory,
                             output_directory=output_directory,
                             output_file_prefix=output_file_prefix)
            
        elif norm_method != "analytic_pearson_residuals":
            for norm_factor in norm_factors:
                output_file_prefix = f"{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}"
    
                tk = tokenizers.CellNeighborhoodRankTokenizer(
                        model_input_size=model_input_size,
                        norm_method=norm_method,
                        norm_factor=norm_factor,
                        separate_cell_and_neighborhood_tokens=separate_cell_and_neighborhood_tokens,
                        cell_gene_means_file=cell_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        cell_gene_nzmedians_file=cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_nzmedians_file=neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"),
                        token_dictionary_file=nichejepa_token_dictionary_file,
                        custom_attr_name_dict=custom_attr_name_dict)
                    
                tk.tokenize_data(input_directory=input_directory,
                                 output_directory=output_directory,
                                 output_file_prefix=output_file_prefix)

Tokenizing '../datasets/st_data/silver/nanostring_cosmx_human_brain/nanostring_cosmx_human_brain.h5ad'...
Filtering cells.
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood graph and aggregating counts.
Normalizing gene expression counts.
Creating Hugging Face dataset...


Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

Saving the dataset (0/55 shards):   0%|          | 0/188686 [00:00<?, ? examples/s]

Tokenizing '../datasets/st_data/silver/nanostring_cosmx_human_brain/nanostring_cosmx_human_brain.h5ad'...
Filtering cells.
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood graph and aggregating counts.


### 2.4 Test Tokenizer

In [ ]:
norm_factor = norm_factors[0]
norm_method = "shifted_log"
model_input_size = n_gene_tokens * 2 # n_gene_tokens * 2, 512
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{dataset_name}_{norm_factor}_{norm_method}_{model_input_size}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

In [ ]:
dataset

In [ ]:
print(dataset["gene_tokens_cell"][0][:10])
print(dataset["gene_tokens_neighborhood"][0][:10])
print(dataset["cell_types"][0])
print(dataset["niche_types"][0])
print(dataset["input_ids"][0][:10])
print(dataset["input_ids"][0][int(model_input_size/2):int(model_input_size/2)+10])

### 2.5 Evaluate Tokenizer

#### 2.5.1 Cell Classification

In [20]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

n_gene_tokens = len([token for token in token_dict.keys() if "ENS" in token])

In [21]:
# Get scgpt gene embeddings for NicheJEPA input token ids
def get_classification_tokens(example, top_n):
    example[f"top_{top_n}_cell_tokens"] = [
        token_dict_reverse[token_id].replace("_cell", "")
        for token_id in example["gene_tokens_cell"][:top_n]]
    example[f"top_{top_n}_neighborhood_tokens"] = [
        token_dict_reverse[token_id].replace("_neighborhood", "")
        for token_id in example["gene_tokens_neighborhood"][:top_n]]                  
    return example

In [ ]:
# Test classification token retrieval
top_n = 10
model_input_size = n_gene_tokens * 2 # n_gene_tokens * 2, 512

norm_method = "shifted_log"
norm_factor = "read_depth" # "read_depth", "cell_area"
normalization = f"{norm_factor}_{norm_method}"
    
output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
dataset = dataset.map(get_classification_tokens,
                      fn_kwargs={"top_n": top_n},
                      #cache_file_name=dataset_cache_file,
                     )

print(dataset[f"top_{top_n}_neighbor_cell_tokens"][0])
print("")
print(dataset[f"top_{top_n}_index_cell_tokens"][0])
print("")

top_n = n_gene_tokens
dataset = dataset.map(get_classification_tokens,
                      fn_kwargs={"top_n": top_n},
                      #cache_file_name=dataset_cache_file,
                     )

print(dataset[f"top_{top_n}_neighbor_cell_tokens"][0])
print("")
print(dataset[f"top_{top_n}_index_cell_tokens"][0])
print("")

##### 2.5.1.1 Baseline (Random Embeddings)

Task: cell and niche type classification using randomly embedded top ranked genes.

In [ ]:
emsemblid_embedding_mapping = {}
for key in token_dict.keys():
   emsemblid_embedding_mapping[key] = np.random.randn(512)

In [ ]:
model_input_size = n_gene_tokens * 2 # n_gene_tokens * 2, 512
scores = []

os.makedirs(f"{results_folder_path}/{tokenizer}/{dataset_name}", exist_ok=True)

for norm_method in ["analytic_pearson_residuals",
                    "mean",
                    "nzmedian",
                    "seurat_v3",
                    "shifted_logmean",
                    "shifted_log",
                   ]:
    for norm_factor in norm_factors:
        if norm_method == "analytic_pearson_residuals":
            normalization = f"{norm_method}"
            if norm_factor != "read_depth":
                continue
        if norm_method != "analytic_pearson_residuals":
            normalization = f"{norm_factor}_{norm_method}"
        
        output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
        tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
        dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

        top_n = n_gene_tokens # n_gene_tokens, 30, 10
        
        dataset = dataset.map(get_classification_tokens,
                              fn_kwargs={"top_n": top_n},
                              #cache_file_name=dataset_cache_file,
                             )
    
        X_cell = np.array(
            [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
             sequence in dataset[f"top_{top_n}_cell_tokens"]])
        X_cell = np.mean(X_cell, axis=1) # aggregate across tokens
        X_cell = X_cell.reshape(X_cell.shape[0], -1)
    
        X_neighborhood = np.array(
            [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
             sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
        X_neighborhood = np.mean(X_neighborhood, axis=1) # aggregate across tokens
        X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)
    
        for token_type in ["cell",
                           "neighborhood"
                          ]:
            for label_cat in label_cats:
                label_encoder = LabelEncoder()
                labels = np.array(
                    label_encoder.fit_transform(dataset[label_cat]))
    
                for random_state in range(10):
    
                    np.random.seed(random_state)
                
                    indices = np.arange(X_cell.shape[0])
                    np.random.shuffle(indices)
                    num_train = int(0.8 * len(indices))
                    train_indices = indices[:num_train]
                    test_indices = indices[num_train:]
                    labels_train = labels[train_indices]
                    labels_test = labels[test_indices]
                    if token_type == "cell":
                        X = X_cell
                    else:
                        X = X_neighborhood
                    X_train = X[train_indices]
                    X_test = X[test_indices]
                    
                    lr_clf = LogisticRegression()
                    lr_clf.fit(X_train, labels_train)
                    train_predictions = lr_clf.predict(X_train)
                    train_f1 = f1_score(labels_train, train_predictions, average="weighted")
                    train_score = lr_clf.score(X_train, labels_train)
                    test_predictions = lr_clf.predict(X_test)
                    test_f1 = f1_score(labels_test, test_predictions, average="weighted")
                    test_score = lr_clf.score(X_test, labels_test)
                    scores.append({"tokenizer": tokenizer,
                                   "normalization": normalization,
                                   "token_type": token_type,
                                   "clf_task": label_cat,
                                   "random_state": random_state,
                                   "clf_category": "train",
                                   "f1_score": train_f1,
                                   "accuracy": train_score})
                    scores.append({"tokenizer": tokenizer,
                                   "normalization": normalization,
                                   "token_type": token_type,
                                   "clf_task": label_cat,
                                   "random_state": random_state,
                                   "clf_category": "test",
                                   "f1_score": test_f1,
                                   "accuracy": test_score})
        
                    scores_df = pd.DataFrame(scores)
                    scores_df.to_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_baseline_top{top_n}.csv", index=False)

##### 2.5.1.1 scGPT

Task: cell and niche type classification using scGPT embedded top ranked genes.

In [7]:
if "sim" not in dataset_name:
    ### Get scGPT gene embeddings for each gene Ensembl id ###
    
    # Set config
    pad_token = "<pad>"
    special_tokens = [pad_token, "<cls>", "<eoc>"]
    pad_value = -2
    n_input_bins = 51
    
    # Get vocab
    scgpt_vocab = GeneVocab.from_file(scgpt_vocab_file)
    for s in special_tokens:
        if s not in scgpt_vocab:
            scgpt_vocab.append_token(s)
    
    # Retrieve model parameters from config files
    with open(scgpt_model_config_file, "r") as f:
        model_configs = json.load(f)
    print(f"Resume model from {scgpt_model_file}, the model args will override the "
          f"config {scgpt_model_config_file}.")
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]
    
    # Initialize model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    ntokens = len(scgpt_vocab)  # size of vocabulary
    model = TransformerModel(ntokens,
                             embsize,
                             nhead,
                             d_hid,
                             nlayers,
                             vocab=scgpt_vocab,
                             pad_value=pad_value,
                             n_input_bins=n_input_bins,)
    
    try:
        model.load_state_dict(torch.load(scgpt_model_file))
        print(f"Loading all model params from {scgpt_model_file}")
    except:
        model_dict = model.state_dict()
        pretrained_dict = torch.load(scgpt_model_file)
        pretrained_dict = {k: v
                           for k, v in pretrained_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
        for k, v in pretrained_dict.items():
            print(f"Loading params {k} with shape {v.shape}")
            model_dict.update(pretrained_dict)
            model.load_state_dict(model_dict)
    model.to(device)
    
    # Create mapping from Ensembl id to scGPT token id
    gene2idx = scgpt_vocab.get_stoi()
    
    metadata_df = pd.read_csv(scgpt_metadata_file)
    
    ensemblid_mapping = dict(zip(metadata_df["feature_name"], metadata_df["feature_id"]))
    ensemblid2idx = {ensemblid_mapping[k]: v for k, v in gene2idx.items() if k in ensemblid_mapping}
    
    # Retrieve the data-independent gene embeddings from scGPT
    gene_ids = np.array([id for id in ensemblid2idx.values()])
    gene_embeddings = model.encoder(torch.tensor(gene_ids, dtype=torch.long).to(device))
    gene_embeddings = gene_embeddings.detach().cpu().numpy()
    
    # Store gene embeddings with Ensembl id
    emsemblid_embedding_mapping = {}
    for i, (key, val) in enumerate(ensemblid2idx.items()):
        emsemblid_embedding_mapping[key] = gene_embeddings[i, :]

Resume model from ../artifacts/scgpt/pretrained_model/best_model.pt, the model args will override the config ../artifacts/scgpt/pretrained_model/args.json.
Loading params encoder.embedding.weight with shape torch.Size([60697, 512])
Loading params encoder.enc_norm.weight with shape torch.Size([512])
Loading params encoder.enc_norm.bias with shape torch.Size([512])
Loading params value_encoder.linear1.weight with shape torch.Size([512, 1])
Loading params value_encoder.linear1.bias with shape torch.Size([512])
Loading params value_encoder.linear2.weight with shape torch.Size([512, 512])
Loading params value_encoder.linear2.bias with shape torch.Size([512])
Loading params value_encoder.norm.weight with shape torch.Size([512])
Loading params value_encoder.norm.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.weight with shape torch.Size([512, 512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.bias with shape torch.Size([512])

In [ ]:
if "sim" not in dataset_name:
    # Test scGPT embedding retrieval
    X_cell = np.array(
        [[emsemblid_embedding_mapping[token] for token in sequence] for
         sequence in dataset[f"top_{top_n}_cell_tokens"]])
    X_cell = X_cell.reshape(X_cell.shape[0], -1)
    
    X_neighborhood = np.array(
        [[emsemblid_embedding_mapping[token] for token in sequence] for
         sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
    X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)
    
    print(X_cell)

In [13]:
if "sim" not in dataset_name:
    model_input_size = n_gene_tokens * 2 # n_gene_tokens * 2, 512
    scores = []
    
    os.makedirs(f"{results_folder_path}/{tokenizer}/{dataset_name}", exist_ok=True)
    
    for norm_method in ["analytic_pearson_residuals",
                        "mean",
                        "nzmedian",
                        "seurat_v3",
                        "shifted_logmean",
                        "shifted_log",
                       ]:
        for norm_factor in norm_factors:
            if norm_method == "analytic_pearson_residuals":
                normalization = f"{norm_method}"
                if norm_factor == "cell_area":
                    continue
            if norm_method != "analytic_pearson_residuals":
                normalization = f"{norm_factor}_{norm_method}"
            
            output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
            tokenized_data_folder_path = f"{st_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
            dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
    
            top_n = 10 # n_gene_tokens, 30, 10
            
            dataset = dataset.map(get_classification_tokens,
                                  fn_kwargs={"top_n": top_n},
                                  #cache_file_name=dataset_cache_file,
                                 )
        
            X_cell = np.array(
                [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
                 sequence in dataset[f"top_{top_n}_cell_tokens"]])
            X_cell = np.mean(X_cell, axis=1) # aggregate across tokens
            X_cell = X_cell.reshape(X_cell.shape[0], -1)
        
            X_neighborhood = np.array(
                [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
                 sequence in dataset[f"top_{top_n}_neighborhood_tokens"]])
            X_neighborhood = np.mean(X_neighborhood, axis=1) # aggregate across tokens
            X_neighborhood = X_neighborhood.reshape(X_neighborhood.shape[0], -1)
        
            for token_type in ["cell",
                               "neighborhood"
                              ]:
                for label_cat in label_cats:
                    label_encoder = LabelEncoder()
                    labels = np.array(
                        label_encoder.fit_transform(dataset[label_cat]))
        
                    for random_state in range(10):
        
                        np.random.seed(random_state)
                    
                        indices = np.arange(X_cell.shape[0])
                        np.random.shuffle(indices)
                        num_train = int(0.8 * len(indices))
                        train_indices = indices[:num_train]
                        test_indices = indices[num_train:]
                        labels_train = labels[train_indices]
                        labels_test = labels[test_indices]
                        if token_type == "cell":
                            X = X_cell
                        else:
                            X = X_neighborhood
                        X_train = X[train_indices]
                        X_test = X[test_indices]
                        
                        lr_clf = LogisticRegression()
                        lr_clf.fit(X_train, labels_train)
                        train_predictions = lr_clf.predict(X_train)
                        train_f1 = f1_score(labels_train, train_predictions, average="weighted")
                        train_score = lr_clf.score(X_train, labels_train)
                        test_predictions = lr_clf.predict(X_test)
                        test_f1 = f1_score(labels_test, test_predictions, average="weighted")
                        test_score = lr_clf.score(X_test, labels_test)
                        scores.append({"tokenizer": tokenizer,
                                       "normalization": normalization,
                                       "token_type": token_type,
                                       "clf_task": label_cat,
                                       "random_state": random_state,
                                       "clf_category": "train",
                                       "f1_score": train_f1,
                                       "accuracy": train_score})
                        scores.append({"tokenizer": tokenizer,
                                       "normalization": normalization,
                                       "token_type": token_type,
                                       "clf_task": label_cat,
                                       "random_state": random_state,
                                       "clf_category": "test",
                                       "f1_score": test_f1,
                                       "accuracy": test_score})
            
                        scores_df = pd.DataFrame(scores)
                        scores_df.to_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt_top{top_n}.csv", index=False)

Loading dataset from disk:   0%|          | 0/55 [00:00<?, ?it/s]

Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

Loading dataset from disk:   0%|          | 0/55 [00:00<?, ?it/s]

Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

Loading dataset from disk:   0%|          | 0/55 [00:00<?, ?it/s]

Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

... (24 outputs left out)

Loading dataset from disk:   0%|          | 0/55 [00:02<?, ?it/s]

Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

Loading dataset from disk:   0%|          | 0/55 [00:00<?, ?it/s]

Map:   0%|          | 0/188686 [00:00<?, ? examples/s]

### 2.6 Miscellaneous

In [ ]:
### Consolodidate all metrics into one file ###

# Define the folder containing CSV files
folder_path = "../datasets/results/cell_neighborhood_tokenizer/nanostring_cosmx_human_brain"

# List all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if "clf_scores" in f and f.endswith('.csv')]

# Initialize an empty list to hold DataFrames
dataframes = []

# Load each CSV file into a DataFrame and append to the list
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all DataFrames vertically
combined_df = pd.concat(dataframes, ignore_index=True)

# Define the order of categories for each column
category_order_col1 = ['analytic_pearson_residuals',
                       'read_depth_mean',
                       'read_depth_nzmedian',
                       'read_depth_seurat_v3',
                       "read_depth_shifted_logmean",
                       "read_depth_shifted_log",
                       "cell_area_mean",
                       "cell_area_nzmedian",
                       "cell_area_seurat_v3",
                       "cell_area_shifted_logmean",
                       "cell_area_shifted_log"]  # Example order for column 1
category_order_col2 = ['cell', 'neighborhood']            # Example order for column 2
category_order_col3 = ['cell_types', 'niche_types']     # Example order for column 3

# Convert columns to categorical with defined order
combined_df['normalization'] = pd.Categorical(combined_df['normalization'], categories=category_order_col1, ordered=True)
combined_df['token_type'] = pd.Categorical(combined_df['token_type'], categories=category_order_col2, ordered=True)
combined_df['clf_task'] = pd.Categorical(combined_df['clf_task'], categories=category_order_col3, ordered=True)

# Sort the DataFrame by the three columns
sorted_df = combined_df.sort_values(by=['normalization', 'token_type', 'clf_task', "random_state"])
sorted_df.reset_index(inplace=True, drop=True)
unique_df = sorted_df.drop_duplicates(subset=['normalization', 'token_type', 'clf_task', 'random_state', 'clf_category'])
unique_df.reset_index(inplace=True, drop=True)

# Display the DataFrame
display(unique_df)

unique_df.to_csv(f"{results_folder_path}/{tokenizer}/{dataset_name}/clf_scores_scgpt.csv", index=False)

In [ ]:
len(unique_df)

In [ ]:
print(np.var([2, 4, 8]))
print(np.var([20, 40, 80]))
print(np.var([2, 4, 8] - np.array([np.mean([2, 20]), np.mean([4, 40]), np.mean([8, 80])])))
print(np.var([20, 40, 80] - np.array([np.mean([2, 20]), np.mean([4, 40]), np.mean([8, 80])])))

In [ ]:
print(np.var(np.log([2, 4, 8])))
print(np.var(np.log([20, 40, 80])))
print(np.var(np.log([2, 4, 8]) - np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))
print(np.var(np.log([20, 40, 80]) - np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))

In [ ]:
print(np.var(np.log([2, 4, 8])))
print(np.var(np.log([20, 40, 80])))
print(np.var(np.log([2, 4, 8]) / np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))
print(np.var(np.log([20, 40, 80]) / np.array([np.mean(np.log([2, 20])), np.mean(np.log([4, 40])), np.mean(np.log([8, 80]))])))